# DD-PRiSM-plus — Step 2: preprocessing

Turns the raw downloads into training-ready tables. **CPU session** — no GPU
needed, and none should be spent here.

**Attach step 1's output first:** right panel → **Add Input → Your Work →**
**Notebook Output**, pick your `01_setup_and_data` version.

Every stage prints the paper's expected count next to the actual one. The two
that decide whether this worked:

| table | expected |
|---|---|
| NCI60 training rows | **7,915,900** |
| combination training rows | **1,387,317** |

In [ ]:
import os, glob, subprocess

REPO = '/kaggle/working/ddprism-plus'
OUT  = '/kaggle/working/processed'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone --quiet https://github.com/SanaNiroomand/DD-PRiSM-plus.git {REPO}
os.chdir(REPO)

# Find step 1's data wherever Kaggle mounted it.
hits = glob.glob('/kaggle/input/**/DOSERESP.zip', recursive=True)
if not hits:
    raise SystemExit('DOSERESP.zip not found under /kaggle/input -- attach '
                     'the output of 01_setup_and_data via Add Input.')
DATA = os.path.dirname(hits[0])
print('data :', DATA)
print('files:', len(os.listdir(DATA)))

## Install

In [ ]:
!pip install --quiet zipfile-deflate64 rdkit pyarrow
print('installed')

## Verify the inputs before spending time on them

In [ ]:
!python scripts/get_data.py --dest {DATA} --check

## Run preprocessing

Roughly 10-20 minutes. DOSERESP is read in chunks, six columns at tight
dtypes, so peak memory stays in the hundreds of MB rather than the 11.1 GB a
naive full read would cost.

In [ ]:
!python scripts/preprocess.py --data {DATA} --out {OUT}

The `almanac_splits` stage above builds the held-out sets Tables S3 and S4
describe. Training refuses to run without them, because the unsplit tables
contain the very rows the paper scores on.

## What came out

In [ ]:
!ls -la {OUT} && du -sh {OUT}

## Save it

**Save Version → Save & Run All (Commit)**, then attach this output to the
training notebook.

If a count is off, send me the numbers — every stage prints what the paper
expected, so a mismatch says which step drifted.